# Carotid Artery Segmentation: Clean Reproducible Pipeline

This notebook provides a reproducible comparison of **Attention U-Net** and **Vision Transformer (ViT)** for carotid wall segmentation.

## Scope
- Primary outcome: segmentation quality (Dice, Jaccard, Mean IoU)
- Secondary: qualitative overlays and confusion matrices
- Optional: derived IMT proxy from masks (research-only)

## Ethics note
Segmentation metrics are the primary validation target. Any IMT values derived from segmentation geometry are **research-oriented estimates** and depend on spacing/boundary assumptions; they are **not standalone clinical diagnostics** without external calibration and clinical validation.

In [ ]:
# 1) Imports and reproducibility
import os
import glob
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow.keras.backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# 2) Data paths and deterministic split
# Adjust DATASET_BASE if needed
DATASET_BASE = Path('/content/data/Common Carotid Artery Ultrasound Images')
image_dir = DATASET_BASE / 'US images'
mask_dir = DATASET_BASE / 'Expert mask images'

image_paths = sorted(glob.glob(str(image_dir / '*.png')))
mask_paths = sorted(glob.glob(str(mask_dir / '*.png')))

assert len(image_paths) == len(mask_paths) and len(image_paths) > 0, 'Check image/mask paths.'

pairs = list(zip(image_paths, mask_paths))

train_pairs, temp_pairs = train_test_split(pairs, test_size=0.30, random_state=SEED, shuffle=True)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.50, random_state=SEED, shuffle=True)

print(f'Total: {len(pairs)} | Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

In [ ]:
# 3) Preprocessing pipelines
IMG_SIZE = 256

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
])


def _load_rgb_gray(img_path, mask_path):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    msk = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        raise ValueError(f'Failed reading: {img_path} | {mask_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    msk = (msk.astype(np.float32) / 255.0)
    return img, msk


def _pad_resize(img, msk):
    h, w, _ = img.shape
    md = max(h, w)
    ph0 = (md - h) // 2
    ph1 = md - h - ph0
    pw0 = (md - w) // 2
    pw1 = md - w - pw0
    img = np.pad(img, ((ph0, ph1), (pw0, pw1), (0, 0)), mode='constant', constant_values=0)
    msk = np.pad(msk, ((ph0, ph1), (pw0, pw1)), mode='constant', constant_values=0)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return img, msk


def _preprocess(img_path, mask_path, use_aug=False):
    img, msk = _load_rgb_gray(img_path.decode(), mask_path.decode())
    img, msk = _pad_resize(img, msk)
    if use_aug:
        out = train_aug(image=img, mask=msk)
        img, msk = out['image'], out['mask']
    return img.astype(np.float32), msk[..., None].astype(np.float32)


def tf_preprocess_train(img_path, mask_path):
    x, y = tf.numpy_function(lambda a, b: _preprocess(a, b, True), [img_path, mask_path], [tf.float32, tf.float32])
    x.set_shape([IMG_SIZE, IMG_SIZE, 3])
    y.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return x, y


def tf_preprocess_eval(img_path, mask_path):
    x, y = tf.numpy_function(lambda a, b: _preprocess(a, b, False), [img_path, mask_path], [tf.float32, tf.float32])
    x.set_shape([IMG_SIZE, IMG_SIZE, 3])
    y.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return x, y

In [ ]:
# 4) tf.data builders
BATCH_TRAIN = 8
BATCH_EVAL = 4

def to_ds(pairs, train=False):
    imgs = [p[0] for p in pairs]
    msks = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((imgs, msks))
    if train:
        ds = ds.shuffle(len(pairs), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(tf_preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(BATCH_TRAIN)
    else:
        ds = ds.map(tf_preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(BATCH_EVAL)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = to_ds(train_pairs, train=True)
val_ds = to_ds(val_pairs, train=False)
test_ds = to_ds(test_pairs, train=False)

print('Datasets ready.')

In [ ]:
# 5) Metrics/loss (single canonical definitions)
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2.0 * inter + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)


def jaccard_index(y_true, y_pred, smooth=1e-6):
    yt = K.flatten(y_true)
    yp = K.flatten(K.round(y_pred))
    inter = K.sum(yt * yp)
    union = K.sum(yt) + K.sum(yp) - inter
    return inter / (union + smooth)


def combined_loss(y_true, y_pred, alpha=0.3):
    bce = K.binary_crossentropy(y_true, y_pred)
    return alpha * bce + (1 - alpha) * dice_loss(y_true, y_pred)

mean_iou_metric = tf.keras.metrics.MeanIoU(num_classes=2, name='mean_iou')

In [ ]:
# 6) Model definitions (ViT + Attention U-Net)
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size
    def call(self, images):
        bs = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        return tf.reshape(patches, [bs, -1, patches.shape[-1]])


class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.proj = layers.Dense(projection_dim)
        self.pos = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)
    def call(self, patch):
        positions = tf.range(start=0, limit=tf.shape(patch)[1], delta=1)
        return self.proj(patch) + self.pos(positions)


def build_vit_segmentation_model(input_shape=(256, 256, 3), patch_size=16, projection_dim=64, num_blocks=4, num_heads=4):
    inp = layers.Input(shape=input_shape)
    num_patches = (input_shape[0] // patch_size) ** 2
    x = Patches(patch_size)(inp)
    x = PatchEncoder(num_patches, projection_dim)(x)
    for _ in range(num_blocks):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        x2 = layers.Add()([x, attn])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(128, activation='gelu')(x3)
        x3 = layers.Dense(projection_dim)(x3)
        x = layers.Add()([x2, x3])

    fm = input_shape[0] // patch_size
    x = layers.Reshape((fm, fm, projection_dim))(x)
    x = layers.Conv2DTranspose(128, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(x)
    out = layers.Conv2D(1, 1, activation='sigmoid', padding='same')(x)
    return tf.keras.Model(inp, out, name='vit_seg')


class EncoderBlock(tf.keras.layers.Layer):
    def __init__(self, filters, dropout_rate=0.1, pooling=True, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = layers.Conv2D(filters, 3, activation='relu', padding='same', kernel_initializer='he_normal')
        self.drop = layers.Dropout(dropout_rate)
        self.conv2 = layers.Conv2D(filters, 3, activation='relu', padding='same', kernel_initializer='he_normal')
        self.pool = layers.MaxPool2D() if pooling else None
    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.drop(x, training=training)
        x = self.conv2(x)
        return (self.pool(x), x) if self.pool else x


class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, filters, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.up = layers.UpSampling2D()
        self.encoder = EncoderBlock(filters, dropout_rate, pooling=False)
    def call(self, inputs, training=False):
        x, skip = inputs
        x = self.up(x)
        if x.shape[1] != skip.shape[1] or x.shape[2] != skip.shape[2]:
            x = tf.image.resize(x, (skip.shape[1], skip.shape[2]))
        x = layers.concatenate([x, skip], axis=3)
        return self.encoder(x, training=training)


class AttentionGate(layers.Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)
        self.wg = layers.Conv2D(filters, 1, padding='same')
        self.wx = layers.Conv2D(filters, 1, padding='same')
        self.psi = layers.Conv2D(1, 1, activation='sigmoid', padding='same')
        self.bn = layers.BatchNormalization()
    def call(self, inputs, training=None):
        g, x = inputs
        g1 = self.wg(g)
        x1 = self.wx(x)
        if g1.shape[1] != x1.shape[1] or g1.shape[2] != x1.shape[2]:
            g1 = tf.image.resize(g1, (x1.shape[1], x1.shape[2]))
        psi = layers.Activation('relu')(layers.Add()([g1, x1]))
        psi = self.psi(psi)
        out = layers.Multiply()([x, psi])
        return self.bn(out, training=training)


def build_attention_unet(input_shape=(256, 256, 3)):
    inp = layers.Input(shape=input_shape)
    p1, c1 = EncoderBlock(16, 0.1, name='Encoder1')(inp)
    p2, c2 = EncoderBlock(32, 0.1, name='Encoder2')(p1)
    p3, c3 = EncoderBlock(64, 0.2, name='Encoder3')(p2)
    p4, c4 = EncoderBlock(128, 0.2, name='Encoder4')(p3)
    enc = EncoderBlock(256, 0.3, pooling=False, name='Encoding')(p4)

    a1 = AttentionGate(128, name='Attention1')([enc, c4])
    d1 = DecoderBlock(128, 0.2, name='Decoder1')([enc, a1])
    a2 = AttentionGate(64, name='Attention2')([d1, c3])
    d2 = DecoderBlock(64, 0.2, name='Decoder2')([d1, a2])
    a3 = AttentionGate(32, name='Attention3')([d2, c2])
    d3 = DecoderBlock(32, 0.1, name='Decoder3')([d2, a3])
    a4 = AttentionGate(16, name='Attention4')([d3, c1])
    d4 = DecoderBlock(16, 0.1, name='Decoder4')([d3, a4])

    out = layers.Conv2D(1, 1, activation='sigmoid', padding='same', name='last_conv')(d4)
    return tf.keras.Model(inp, out, name='attention_unet')

In [ ]:
# 7) Train helper and training runs
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger

def compile_model(m):
    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=combined_loss,
        metrics=['accuracy', dice_coefficient, jaccard_index, mean_iou_metric],
    )


def fit_model(m, name, epochs=30):
    cbs = [
        ModelCheckpoint(f'{name}.keras', monitor='val_dice_coefficient', mode='max', save_best_only=True),
        EarlyStopping(monitor='val_dice_coefficient', mode='max', patience=10, restore_best_weights=True),
        CSVLogger(f'{name}_training_log.csv', append=False),
    ]
    return m.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs)

vit_model = build_vit_segmentation_model()
unet_model = build_attention_unet()

compile_model(vit_model)
compile_model(unet_model)

vit_history = fit_model(vit_model, 'AttentionViT', epochs=30)
unet_history = fit_model(unet_model, 'AttentionUNet', epochs=30)

In [ ]:
# 8) Final validation comparison table + bar chart
import pandas as pd

uh = unet_history.history
vh = vit_history.history

comparison = pd.DataFrame([
    {
        'Model': 'Attention U-Net',
        'Loss': uh['val_loss'][-1],
        'Accuracy': uh['val_accuracy'][-1],
        'Dice': uh['val_dice_coefficient'][-1],
        'Jaccard': uh['val_jaccard_index'][-1],
        'MeanIoU': uh['val_mean_iou'][-1],
    },
    {
        'Model': 'Vision Transformer',
        'Loss': vh['val_loss'][-1],
        'Accuracy': vh['val_accuracy'][-1],
        'Dice': vh['val_dice_coefficient'][-1],
        'Jaccard': vh['val_jaccard_index'][-1],
        'MeanIoU': vh['val_mean_iou'][-1],
    },
])
comparison['Absolute Difference'] = np.nan
comparison.loc[len(comparison)] = ['Absolute Difference',
                                   abs(comparison.loc[0, 'Loss'] - comparison.loc[1, 'Loss']),
                                   abs(comparison.loc[0, 'Accuracy'] - comparison.loc[1, 'Accuracy']),
                                   abs(comparison.loc[0, 'Dice'] - comparison.loc[1, 'Dice']),
                                   abs(comparison.loc[0, 'Jaccard'] - comparison.loc[1, 'Jaccard']),
                                   abs(comparison.loc[0, 'MeanIoU'] - comparison.loc[1, 'MeanIoU']),
                                   np.nan]

comparison

# Dice chart
plt.figure(figsize=(8, 5))
models = ['Attention U-Net', 'Vision Transformer']
dice_vals = [uh['val_dice_coefficient'][-1], vh['val_dice_coefficient'][-1]]
colors = ['#4c72b0', '#55a868']
plt.bar(models, dice_vals, color=colors)
for i, v in enumerate(dice_vals):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center')
plt.ylim(0, 1.0)
plt.ylabel('Validation Dice')
plt.title('Validation Dice: U-Net vs ViT')
plt.show()

In [ ]:
# 9) Test-set confusion matrices
# Collect test tensors
X_list, y_list = [], []
for xb, yb in test_ds:
    X_list.append(xb.numpy())
    y_list.append(yb.numpy())
X_test = np.concatenate(X_list, axis=0)
y_test = np.concatenate(y_list, axis=0)

y_true = (y_test > 0.5).astype(np.uint8).ravel()
y_pred_unet = (unet_model.predict(X_test, verbose=0) > 0.5).astype(np.uint8).ravel()
y_pred_vit = (vit_model.predict(X_test, verbose=0) > 0.5).astype(np.uint8).ravel()

cm_unet = confusion_matrix(y_true, y_pred_unet)
cm_vit = confusion_matrix(y_true, y_pred_vit)

labels = ['Background', 'Vessel Wall']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
import seaborn as sns
sns.heatmap(cm_unet, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Attention U-Net Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Vision Transformer Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# 10) Optional qualitative sample panel
n = min(4, len(X_test))
idxs = np.arange(n)

pred_u = (unet_model.predict(X_test[idxs], verbose=0) > 0.5).astype(np.uint8)
pred_v = (vit_model.predict(X_test[idxs], verbose=0) > 0.5).astype(np.uint8)

fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    axes = np.expand_dims(axes, 0)
for r, i in enumerate(idxs):
    axes[r, 0].imshow(X_test[i])
    axes[r, 0].set_title('Original')
    axes[r, 0].axis('off')

    axes[r, 1].imshow(X_test[i])
    axes[r, 1].imshow(y_test[i].squeeze(), cmap='jet', alpha=0.4)
    axes[r, 1].set_title('Ground Truth')
    axes[r, 1].axis('off')

    axes[r, 2].imshow(X_test[i])
    axes[r, 2].imshow(pred_u[r].squeeze(), cmap='viridis', alpha=0.45)
    axes[r, 2].set_title('Attention U-Net')
    axes[r, 2].axis('off')

    axes[r, 3].imshow(X_test[i])
    axes[r, 3].imshow(pred_v[r].squeeze(), cmap='plasma', alpha=0.45)
    axes[r, 3].set_title('Vision Transformer')
    axes[r, 3].axis('off')
plt.tight_layout()
plt.show()

## Reporting guidance

- Treat Dice/Jaccard/IoU as the primary model validation for segmentation.
- If deriving IMT from masks, label it as a **derived proxy** unless calibrated with validated spacing and boundary protocol.
- For deployment recommendation, prefer the model with consistently better overlap metrics and lower clinically relevant error profile (FN/FP balance from confusion matrices).